# Tự động tạo Dataset & Train TinyML Anomaly Detector

Mô hình được tối ưu hóa cho các ngưỡng:
- **Temp Normal:** 25.0 - 30.0 °C
- **Humi Normal:** 40.0 - 60.0 %

**Quy trình:**
1. Sinh dữ liệu giả lập (Synthetic Data) theo ngưỡng trên.
2. Huấn luyện mô hình Autoencoder (Input 2 -> Hidden 8 -> Output 2).
3. Kiểm tra độ chính xác.
4. Xuất file Header C++.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# --- 1. SINH DỮ LIỆU GIẢ LẬP (SYNTHETIC DATA) ---
np.random.seed(42)
num_samples = 2000

# Tạo dữ liệu Bình thường (Anomaly = 0)
normal_temp = np.random.uniform(25.0, 30.0, int(num_samples * 0.8))
normal_humi = np.random.uniform(40.0, 60.0, int(num_samples * 0.8))
normal_labels = np.zeros(int(num_samples * 0.8))

# Tạo dữ liệu Bất thường (Anomaly = 1)
# Case 1: Quá nóng, Case 2: Quá lạnh, Case 3: Quá ẩm, Case 4: Quá khô
anom_temp = np.concatenate([
    np.random.uniform(35.0, 45.0, 100), # Nóng
    np.random.uniform(10.0, 20.0, 100), # Lạnh
    np.random.uniform(25.0, 30.0, 200)  # Temp bình thường nhưng Humi sai
])
anom_humi = np.concatenate([
    np.random.uniform(20.0, 80.0, 200),
    np.random.uniform(70.0, 95.0, 100), # Quá ẩm
    np.random.uniform(10.0, 30.0, 100)  # Quá khô
])
anom_labels = np.ones(len(anom_temp))

# Gộp lại thành DataFrame
df = pd.DataFrame({
    'Temperature': np.concatenate([normal_temp, anom_temp]),
    'Humidity': np.concatenate([normal_humi, anom_humi]),
    'Anomaly': np.concatenate([normal_labels, anom_labels])
})

df = df.sample(frac=1).reset_index(drop=True) # Trộn dữ liệu
df.to_csv('custom_iot_dataset.csv', index=False)
print("Đã tạo file custom_iot_dataset.csv thành công!")

# --- 2. CHUẨN HÓA ---
X_train_normal = df[df['Anomaly'] == 0][['Temperature', 'Humidity']]
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)

print(f"Mean: {scaler.mean_}")
print(f"Scale: {scaler.scale_}")

Đã tạo file custom_iot_dataset.csv thành công!
Mean: [27.49268843 49.99301884]
Scale: [1.46541064 5.78233689]


In [2]:
# --- 3. TRAIN AUTOENCODER ---
model = models.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(2, activation='linear')
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=16, verbose=0)

# Tính Threshold
reconstructions = model.predict(X_train_scaled)
mse = np.mean(np.square(reconstructions - X_train_scaled), axis=1)
threshold = np.mean(mse) + np.std(mse) * 3
print(f"Threshold: {threshold}")

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Threshold: 0.0005891561122366995


In [7]:
# --- 4. KIỂM TRA (TEST) ---
print("\n--- TEST VỚI CÁC NGƯỠNG BIÊN ---")
test_cases = pd.DataFrame([
    [20.5, 40.0], # Bình thường (Giữa khoảng)
    [24.0, 70.0], # Hơi nóng (Bất thường)
    [35.5, 60.0], # Hơi ẩm (Bất thường)
    [15.0, 20.0]  # Rất bất thường
], columns=['Temperature', 'Humidity'])

test_scaled = scaler.transform(test_cases)
test_recons = model.predict(test_scaled)
test_mse = np.mean(np.square(test_recons - test_scaled), axis=1)

for i in range(len(test_cases)):
    res = "ANOMALY" if test_mse[i] > threshold else "Normal"
    print(f"Data: {test_cases.values[i]} -> MSE: {test_mse[i]:.4f} -> {res}")


--- TEST VỚI CÁC NGƯỠNG BIÊN ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
Data: [20.5 40. ] -> MSE: 0.0051 -> ANOMALY
Data: [24. 70.] -> MSE: 0.0065 -> ANOMALY
Data: [35.5 60. ] -> MSE: 0.0008 -> ANOMALY
Data: [15. 20.] -> MSE: 0.0246 -> ANOMALY


In [4]:
# --- 5. EXPORT TO C++ ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

def hex_to_c_array(hex_data, var_name):
    c_str = f"const unsigned char {var_name}[] = {{\n  "
    for i, byte in enumerate(hex_data):
        c_str += f"0x{byte:02x}, "
        if (i + 1) % 12 == 0: c_str += "\n  "
    c_str = c_str.strip(", \n") + "\n};\n"
    c_str += f"const unsigned int {var_name}_len = {len(hex_data)};"
    return c_str

with open("dht_model_v2.h", "w") as f:
    f.write(hex_to_c_array(tflite_model, "dht_anomaly_model_tflite"))

print(f"Model size: {len(tflite_model)} bytes")
print("Success! Copy nội dung dht_model_v2.h vào dự án.")

Saved artifact at '/tmp/tmp2dxouq6t'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  132469988149968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132469988151120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132469988149776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132469988148240: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model size: 1692 bytes
Success! Copy nội dung dht_model_v2.h vào dự án.
